In [1]:
# PHASE 2 - NOTEBOOK D7a: evaluation of the CCSM checkpoints for resnet18, vit_small, swin_tiny
# Kaggle: GPU on, Internet on.  Attach the checkpoints dataset  badrinathachanta/ccsm-checkpoints-1  as input (it must contain the ccsm_<family>_<arm>_s<seed>_best.pt files of these families).
# The originals are downloaded from the model hubs.  RUNTIME ~10-12 h (ResNet18 ~1 h, ViT-S ~4 h, Swin-T ~6 h); resumable: after every model its tiny result .json is saved and the big feature file is deleted.
#   If the 12 h session ends: Save Version -> attach that version's OUTPUT as an input of the next run; finished models are reused and skipped.
!pip install -q timm datasets kagglehub transformers

"""
Frozen linear probe, Part-I recipe (StandardScaler + LogisticRegression C=1 balanced, max_iter 2000; primary split centres 0,1,3,4 -> 2 with 1000
patient-level bootstraps, seed 7; 5-fold leave-one-centre-out; patch-level centre-ID probe).  Phikon is extracted in fp16.
Order inside each family: chunk 1 = original + seed 42 (go/no-go), chunk 2 = seed 43, chunk 3 = seed 44.
Caveat: for LOCO folds 0,1,3,4 the held-out centre's UNLABELED images were in the pretraining pool; centre 2 (primary split) is clean.
"""
import os, gc, json, glob, random, time, shutil, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score
from tqdm import tqdm

warnings.filterwarnings('ignore')
T_START = time.time()


class Config:
    RUN_FAMILIES = ['resnet18', 'vit_small', 'swin_tiny']
    SEEDS = [42, 43, 44]
    BUDGET_H = 11.0                       # stop starting new chunks when the next one would not fit
    HF_DATASET = "wltjr1007/Camelyon17-WILDS"
    CKPT_SEARCH_ROOTS = ["/kaggle/input", "/kaggle/working/checkpoints"]
    KAGGLE_CKPT_DATASETS = ["badrinathachanta/ccsm-checkpoints-1"]   # fallback if not attached
    PRIOR_RESULT_ROOTS = ["/kaggle/input"]                                  # earlier runs' outputs (p2_eval_all/*.json) are reused
    FEATURE_DIR = "/kaggle/working/p2_features_a"
    RESULTS_DIR = "/kaggle/working/p2_eval_a"
    KEEP_FEATURES = False

    EXTRACT_BATCH = 128
    NUM_WORKERS = 4
    ALL_CENTERS = [0, 1, 2, 3, 4]
    TEST_CENTER = 2
    LR_C, LR_MAX_ITER, PROBE_SEED = 1.0, 2000, 42
    BOOT_N, BOOT_SEED = 1000, 7
    LOCO_FOLDS = [0, 1, 2, 3, 4]
    CID_TRAIN_PER_CENTER, CID_TEST_PER_CENTER, CID_VAL_FRAC = 3000, 1000, 0.25

    FAMILIES = {   # kind, hub id / timm name, input size, fp16 extraction, Part-I LOCO of the original (pipeline check; None = ImageNet control)
        'resnet18':  dict(kind='resnet', hub=None, size=96,  amp=False, ref=0.9528, label='ImageNet ResNet18'),
        'vit_small': dict(kind='timm', hub='vit_small_patch16_224', size=224, amp=False, ref=None, label='ImageNet ViT-S/16'),
        'swin_tiny': dict(kind='timm', hub='swin_tiny_patch4_window7_224', size=224, amp=False, ref=None, label='ImageNet Swin-T'),
        'kaiko':     dict(kind='timm', hub='hf-hub:1aurent/vit_small_patch16_224.kaiko_ai_towards_large_pathology_fms', size=224, amp=False, ref=0.9873, label='Kaiko ViT-S/16'),
        'lunit':     dict(kind='timm', hub='hf-hub:1aurent/vit_small_patch16_224.lunit_dino', size=224, amp=False, ref=0.9814, label='Lunit ViT-S/16'),
        'phikon':    dict(kind='hf', hub='owkin/phikon', size=224, amp=True, ref=0.9818, label='Phikon ViT-B/16'),
    }
    ARMS = {'resnet18': ['control', 'ms', 'msrand', 'fms']}
    DEFAULT_ARMS = ['control', 'ms', 'fms']
    PART1_OTHERS = "Part-I LOCO references: Stain Aug 0.9884 | Kaiko 0.9873 | Phikon 0.9818 | Lunit 0.9814 | ImageNet ResNet18 0.9528"

    @staticmethod
    def arms(fam):
        return Config.ARMS.get(fam, Config.DEFAULT_ARMS)

    @staticmethod
    def chunks(fam):
        """[(chunk_models)] : chunk 1 = original + seed 42, then one chunk per further seed.  Model name = '<family>__ctrl' / '<family>__<arm>_s<seed>'."""
        out = []
        for k, s in enumerate(Config.SEEDS):
            c = [f'{fam}__ctrl'] if k == 0 else []
            c += [f'{fam}__{a}_s{s}' for a in Config.arms(fam)]
            out.append(c)
        return out


def parse(name):
    fam, rest = name.split('__')
    if rest == 'ctrl':
        return fam, 'ctrl', None
    arm, s = rest.rsplit('_s', 1)
    return fam, arm, int(s)


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


# ---------------------------------------------------------------- encoders (same construction as Part I / the training notebooks)
class Encoder(nn.Module):
    def __init__(self, fam):
        super().__init__()
        spec = Config.FAMILIES[fam]; self.pool = None
        if spec['kind'] == 'resnet':
            net = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
            self.backbone = nn.Sequential(*list(net.children())[:-1]); self.pool = nn.AdaptiveAvgPool2d(1); self.dim = 512
        elif spec['kind'] == 'hf':
            from transformers import AutoModel
            class PhikonBackbone(nn.Module):
                def __init__(self):
                    super().__init__()
                    self.vit = AutoModel.from_pretrained(spec['hub'])
                def forward(self, x):
                    return self.vit(pixel_values=x).last_hidden_state[:, 0]           # CLS token, as in Part I
            self.backbone = PhikonBackbone(); self.dim = self.backbone.vit.config.hidden_size
        else:
            import timm
            self.backbone = timm.create_model(spec['hub'], pretrained=True, num_classes=0)
            self.dim = self.backbone.num_features

    @torch.no_grad()
    def encode(self, x):
        f = self.backbone(x)
        if f.dim() == 4:
            f = torch.flatten(self.pool(f), 1)
        elif f.dim() == 3:
            f = f.mean(1)
        return f


def ensure_checkpoint_roots():
    if glob.glob("/kaggle/input/**/ccsm_*_best.pt", recursive=True):
        return
    import kagglehub
    for slug in Config.KAGGLE_CKPT_DATASETS:
        try:
            path = kagglehub.dataset_download(slug); print("✓ checkpoints downloaded via kagglehub:", path)
            Config.CKPT_SEARCH_ROOTS.append(path)
        except Exception as e:
            print(f"⚠ could not download {slug}: {e}")


def find_checkpoint(fam, arm, seed):
    fname = f"ccsm_{fam}_{arm}_s{seed}_best.pt"
    for root in Config.CKPT_SEARCH_ROOTS:
        hits = glob.glob(os.path.join(root, '**', fname), recursive=True)
        if hits:
            return hits[0]
    return None


def build_model(name):
    fam, arm, seed = parse(name)
    if arm == 'ctrl':
        return Encoder(fam)
    path = find_checkpoint(fam, arm, seed)
    if path is None:
        return None
    enc = Encoder(fam)
    ck = torch.load(path, map_location='cpu')
    missing, unexpected = enc.load_state_dict(ck['model_state_dict'], strict=False)
    bad = [k for k in unexpected if not k.startswith(('head.', 'adv_head.'))]
    assert not missing and not bad, f"{name}: missing={missing[:3]} unexpected={bad[:3]}"
    print(f"  [{name}] loaded {os.path.basename(path)}  (epoch {ck.get('epoch','?')}, final loss {ck.get('best_val', float('nan')):.4f})")
    return enc


# ---------------------------------------------------------------- one-pass feature extraction
class MultiSizeDataset(Dataset):
    def __init__(self, hf, sizes):
        self.hf, self.sizes = hf, sorted(sizes)
        norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        self.tf = {s: transforms.Compose([transforms.Resize((s, s)), transforms.ToTensor(), norm]) for s in self.sizes}

    def __len__(self):
        return len(self.hf)

    def __getitem__(self, i):
        img = self.hf[int(i)]['image']
        if img.mode != 'RGB':
            img = img.convert('RGB')
        return {s: self.tf[s](img) for s in self.sizes}


def feat_path(name):
    return f"{Config.FEATURE_DIR}/{name}.npy"


def extract_missing(hf, names, device):
    todo = [n for n in names if not os.path.exists(feat_path(n))]
    if not todo:
        return
    mods = {}
    for n in todo:
        m = build_model(n)
        if m is None:
            print(f"  [{n}] checkpoint not found -> skipped")
            continue
        mods[n] = m.to(device).eval()
    if not mods:
        return
    fam = parse(todo[0])[0]; size = Config.FAMILIES[fam]['size']; amp = Config.FAMILIES[fam]['amp'] and device.type == 'cuda'
    N = len(hf)
    out = {n: np.empty((N, mods[n].dim), dtype=np.float32) for n in mods}
    loader = DataLoader(MultiSizeDataset(hf, {size}), batch_size=Config.EXTRACT_BATCH, shuffle=False,
                        num_workers=Config.NUM_WORKERS, pin_memory=device.type == 'cuda')
    pos, t0 = 0, time.time()
    for batch in tqdm(loader, desc=f"extract {fam}: {len(mods)} models"):
        x = batch[size].to(device, non_blocking=True); B = x.size(0)
        for n, m in mods.items():
            with torch.autocast('cuda', enabled=amp):
                f = m.encode(x)
            out[n][pos:pos + B] = f.float().cpu().numpy()
        pos += B
    assert pos == N, (pos, N)
    os.makedirs(Config.FEATURE_DIR, exist_ok=True)
    for n in mods:
        np.save(feat_path(n), out[n])
    print(f"✓ extracted {list(mods)} in {(time.time()-t0)/60:.1f} min")
    del mods, out; gc.collect(); torch.cuda.empty_cache()


# ---------------------------------------------------------------- evaluation (same recipe as Part I)
def bootstrap_auroc_by_patient(y, p, pat, n_boot, seed):
    pats = np.unique(pat)
    idx_by = {q: np.where(pat == q)[0] for q in pats}
    rng, vals = np.random.RandomState(seed), []
    for _ in range(n_boot):
        s = rng.choice(pats, size=len(pats), replace=True)
        idx = np.concatenate([idx_by[q] for q in s])
        if len(np.unique(y[idx])) < 2:
            continue
        vals.append(roc_auc_score(y[idx], p[idx]))
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return float(lo), float(hi)


def probe_auroc(X, y, centers, patients, train_c, test_c, with_ci=False):
    tr, te = np.isin(centers, train_c), centers == test_c
    sc = StandardScaler().fit(X[tr])
    clf = LogisticRegression(max_iter=Config.LR_MAX_ITER, C=Config.LR_C, class_weight='balanced',
                             random_state=Config.PROBE_SEED, n_jobs=-1)
    clf.fit(sc.transform(X[tr]), y[tr])
    prob = clf.predict_proba(sc.transform(X[te]))[:, 1]
    auc = roc_auc_score(y[te], prob)
    bal = balanced_accuracy_score(y[te], prob > 0.5)
    if with_ci:
        lo, hi = bootstrap_auroc_by_patient(y[te], prob, patients[te], Config.BOOT_N, Config.BOOT_SEED)
        return auc, bal, lo, hi
    return auc, bal


def center_id_probe(X, centers, patients):
    """Patch-level 5-way center-identity probe, patient-disjoint train/test within each center."""
    rng = np.random.RandomState(Config.PROBE_SEED)
    tr_idx, te_idx = [], []
    for c in Config.ALL_CENTERS:
        pats = sorted(np.unique(patients[centers == c]).tolist())
        random.Random(Config.PROBE_SEED + c).shuffle(pats)
        n_te = min(max(1, int(round(Config.CID_VAL_FRAC * len(pats)))), len(pats) - 1)
        te_p, tr_p = set(pats[:n_te]), set(pats[n_te:])
        for pool, n_want, store in ((np.where((centers == c) & np.isin(patients, list(tr_p)))[0], Config.CID_TRAIN_PER_CENTER, tr_idx),
                                    (np.where((centers == c) & np.isin(patients, list(te_p)))[0], Config.CID_TEST_PER_CENTER, te_idx)):
            store.append(rng.choice(pool, size=min(n_want, len(pool)), replace=False))
    tr_idx, te_idx = np.concatenate(tr_idx), np.concatenate(te_idx)
    sc = StandardScaler().fit(X[tr_idx])
    clf = LogisticRegression(max_iter=Config.LR_MAX_ITER, n_jobs=-1).fit(sc.transform(X[tr_idx]), centers[tr_idx])
    return 100 * accuracy_score(centers[te_idx], clf.predict(sc.transform(X[te_idx])))


def evaluate_model(name, y, centers, patients):
    cache = f"{Config.RESULTS_DIR}/{name}.json"
    if os.path.exists(cache):
        return json.load(open(cache))
    X = np.load(feat_path(name), mmap_mode='r')
    X = np.ascontiguousarray(X)
    t0 = time.time()
    auc, bal, lo, hi = probe_auroc(X, y, centers, patients, [c for c in Config.ALL_CENTERS if c != Config.TEST_CENTER],
                                   Config.TEST_CENTER, with_ci=True)
    loco = {}
    for ho in Config.LOCO_FOLDS:
        loco[ho] = probe_auroc(X, y, centers, patients, [c for c in Config.ALL_CENTERS if c != ho], ho)[0]
    vals = np.array(list(loco.values()))
    res = {'model': name, 'primary_auroc': auc, 'primary_ci_lo': lo, 'primary_ci_hi': hi, 'primary_balacc': bal,
           'loco_mean': float(vals.mean()), 'loco_std': float(vals.std(ddof=1)) if len(vals) > 1 else 0.0,
           'loco_folds': {str(k): float(v) for k, v in loco.items()},
           'center_id_acc': center_id_probe(X, centers, patients)}
    print(f"  [{name}] primary {auc:.4f} [{lo:.4f},{hi:.4f}] | LOCO {res['loco_mean']:.4f}±{res['loco_std']:.4f} | "
          f"center-ID {res['center_id_acc']:.1f}% | {(time.time()-t0)/60:.1f} min")
    os.makedirs(Config.RESULTS_DIR, exist_ok=True)
    json.dump(res, open(cache, 'w'))
    del X; gc.collect()
    if not Config.KEEP_FEATURES and os.path.exists(feat_path(name)):
        os.remove(feat_path(name))
    return res


def _ms(v):
    v = np.array(v, dtype=float)
    return v.mean(), (v.std(ddof=1) if len(v) > 1 else float('nan'))



def load_prior_results():
    os.makedirs(Config.RESULTS_DIR, exist_ok=True)
    n = 0
    for root in Config.PRIOR_RESULT_ROOTS:
        for p in glob.glob(os.path.join(root, '**', 'p2_eval_a', '*.json'), recursive=True):
            dst = os.path.join(Config.RESULTS_DIR, os.path.basename(p))
            if not os.path.exists(dst):
                shutil.copy2(p, dst); n += 1
    if n:
        print(f"✓ reused {n} finished results from earlier runs")


def have_result(name):
    return os.path.exists(f"{Config.RESULTS_DIR}/{name}.json")


def load_meta(hf):
    return (np.array(hf['label']).astype(np.int64), np.array(hf['center']).astype(np.int64),
            np.array(hf['patient']).astype(np.int64))


LAB = {'ctrl': 'ORIGINAL model', 'control': 'SimCLR control (no mixing)', 'ms': 'CCSM (MixStyle, cross-centre)',
       'msrand': 'MixStyle, random partners', 'fms': 'CCSM + Fourier 0.05'}


def summarize():
    res = {}
    for p in glob.glob(f"{Config.RESULTS_DIR}/*.json"):
        r = json.load(open(p)); res[r['model']] = r
    rows, per_all = [], {}
    print("\n" + "=" * 112 + "\nCCSM -- ALL FAMILIES  (frozen linear probe; mean ± std over seeds, ddof=1)\n" + "=" * 112)
    for fam in Config.FAMILIES:
        spec = Config.FAMILIES[fam]
        ctrl = res.get(f'{fam}__ctrl')
        per = {a: {} for a in Config.arms(fam)}
        for n, r in res.items():
            f, a, s = parse(n)
            if f == fam and a != 'ctrl':
                per[a][s] = (r['primary_auroc'], r['loco_mean'], r['center_id_acc'])
        if not ctrl and not any(per.values()):
            continue
        per_all[fam] = per
        print(f"\n--- {fam}  ({spec['label']}) ---")
        if ctrl:
            rows.append(dict(family=fam, method=LAB['ctrl'], seeds='-', LOCO=f"{ctrl['loco_mean']:.4f}", primary=f"{ctrl['primary_auroc']:.4f}", centre_ID=f"{ctrl['center_id_acc']:.1f}"))
            print(f"  {LAB['ctrl']:30s} LOCO {ctrl['loco_mean']:.4f} | primary {ctrl['primary_auroc']:.4f} [{ctrl['primary_ci_lo']:.4f},{ctrl['primary_ci_hi']:.4f}] | centre-ID {ctrl['center_id_acc']:.1f}")
            if spec['ref']:
                ok = abs(ctrl['loco_mean'] - spec['ref']) < (0.005 if spec['amp'] else 0.003)
                print(f"      pipeline check vs Part I ({spec['ref']:.4f}): {'OK' if ok else 'MISMATCH - investigate before trusting this family'}")
        for a, d in per.items():
            if not d: continue
            lm, ls = _ms([v[1] for v in d.values()]); pm, ps = _ms([v[0] for v in d.values()]); cm, cs = _ms([v[2] for v in d.values()])
            rows.append(dict(family=fam, method=LAB[a], seeds=','.join(map(str, sorted(d))), LOCO=f"{lm:.4f} ± {ls:.4f}", primary=f"{pm:.4f} ± {ps:.4f}", centre_ID=f"{cm:.1f} ± {cs:.1f}"))
            print(f"  {LAB[a]:30s} LOCO {lm:.4f} ± {ls:.4f} | primary {pm:.4f} ± {ps:.4f} | centre-ID {cm:.1f} ± {cs:.1f}   per-seed LOCO: " + " ".join(f"s{s}:{d[s][1]:.4f}" for s in sorted(d)))
        def paired(a, b, label):
            if a in per and b in per:
                common = sorted(set(per[a]) & set(per[b]))
                if common:
                    g = [100 * (per[a][s][1] - per[b][s][1]) for s in common]; m, sd = _ms(g)
                    print(f"      {label:36s} " + " ".join(f"s{s}:{x:+.2f}" for s, x in zip(common, g)) + f"   mean {m:+.2f} ± {sd:.2f}  wins {sum(x > 0 for x in g)}/{len(g)}")
        paired('ms', 'control', 'CCSM minus SimCLR control (x100)')
        paired('ms', 'msrand', 'cross-centre minus random partners')
        paired('fms', 'ms', 'Fourier add-on minus CCSM')
        if ctrl:
            for a, d in per.items():
                if d:
                    g = [100 * (v[1] - ctrl['loco_mean']) for v in d.values()]; m, sd = _ms(g)
                    print(f"      gain over ORIGINAL, {a:8s} {m:+.2f} ± {sd:.2f}  (seeds above original: {sum(x > 0 for x in g)}/{len(g)})")
    df = pd.DataFrame(rows)
    os.makedirs(Config.RESULTS_DIR, exist_ok=True)
    df.to_csv(f"{Config.RESULTS_DIR}/ccsm_all_summary.csv", index=False)
    print("\n" + Config.PART1_OTHERS)
    print("Caveat: LOCO folds 0,1,3,4 include the held-out centre's unlabeled images in pretraining; centre 2 (primary) is clean.")
    expected = [n for fam in Config.RUN_FAMILIES for c in Config.chunks(fam) for n in c]
    missing = [n for n in expected if not have_result(n)]
    print(f"\nProgress: {len(expected) - len(missing)}/{len(expected)} models evaluated." + (f" Still missing: {missing[:12]}{' ...' if len(missing) > 12 else ''}" if missing else " ALL DONE."))
    return df


def main(hf=None):
    set_seed(Config.PROBE_SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"device: {device}")
    ensure_checkpoint_roots(); load_prior_results()
    if hf is None:
        from datasets import load_dataset, concatenate_datasets
        ds = load_dataset(Config.HF_DATASET)
        parts = [ds[k] for k in ds.keys()]
        hf = concatenate_datasets(parts) if len(parts) > 1 else parts[0]
    print(f"✓ {len(hf):,} patches")
    y, centers, patients = load_meta(hf)
    os.makedirs(Config.FEATURE_DIR, exist_ok=True)
    last_chunk_h = 0.0
    for fam in Config.RUN_FAMILIES:
        for ci, chunk in enumerate(Config.chunks(fam), 1):
            todo = [n for n in chunk if not have_result(n)]
            if not todo:
                continue
            elapsed = (time.time() - T_START) / 3600
            if elapsed + max(last_chunk_h, 0.3) > Config.BUDGET_H:
                print(f"\n⏸ time budget reached ({elapsed:.1f} h used). Save this version and continue in a new session (attach this output as input).")
                return summarize()
            print(f"\n=== {fam} chunk {ci}: {todo}")
            t0 = time.time()
            extract_missing(hf, todo, device)
            for n in todo:
                if os.path.exists(feat_path(n)):
                    evaluate_model(n, y, centers, patients)
                else:
                    print(f"  [{n}] no features (checkpoint missing) -> not evaluated")
            last_chunk_h = (time.time() - t0) / 3600
            print(f"  chunk took {last_chunk_h:.2f} h | total elapsed {(time.time() - T_START) / 3600:.1f} h")
    return summarize()


# ====================================================================
# RUN
# ====================================================================
results_df = main()


device: cuda


README.md:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

data/train-00000-of-00014.parquet: reconstructing file:   0%|          |  0.00B /  470MB            

data/train-00000-of-00014.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00014.parquet: reconstructing file:   0%|          |  0.00B /  469MB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  114MB            

model.safetensors: downloading bytes:           |  0.00B            

  [swin_tiny__control_s42] loaded ccsm_swin_tiny_control_s42_best.pt  (epoch 6, final loss 0.0990)
  [swin_tiny__ms_s42] loaded ccsm_swin_tiny_ms_s42_best.pt  (epoch 6, final loss 0.1196)
  [swin_tiny__fms_s42] loaded ccsm_swin_tiny_fms_s42_best.pt  (epoch 6, final loss 0.1381)


extract swin_tiny: 4 models: 100%|██████████| 3563/3563 [2:14:03<00:00,  2.26s/it]


✓ extracted ['swin_tiny__ctrl', 'swin_tiny__control_s42', 'swin_tiny__ms_s42', 'swin_tiny__fms_s42'] in 134.1 min
  [swin_tiny__ctrl] primary 0.9735 [0.9306,0.9889] | LOCO 0.9687±0.0061 | center-ID 88.8% | 5.8 min
  [swin_tiny__control_s42] primary 0.9751 [0.9416,0.9855] | LOCO 0.9704±0.0072 | center-ID 92.6% | 7.2 min
  [swin_tiny__ms_s42] primary 0.9821 [0.9607,0.9889] | LOCO 0.9721±0.0114 | center-ID 88.1% | 7.4 min
  [swin_tiny__fms_s42] primary 0.9787 [0.9499,0.9877] | LOCO 0.9676±0.0171 | center-ID 87.0% | 8.0 min
  chunk took 2.71 h | total elapsed 8.5 h

⏸ time budget reached (8.5 h used). Save this version and continue in a new session (attach this output as input).

CCSM -- ALL FAMILIES  (frozen linear probe; mean ± std over seeds, ddof=1)

--- resnet18  (ImageNet ResNet18) ---
  ORIGINAL model                 LOCO 0.9528 | primary 0.9473 [0.9208,0.9596] | centre-ID 78.4
      pipeline check vs Part I (0.9528): OK
  SimCLR control (no mixing)     LOCO 0.8678 ± 0.0130 | primar